# Dataset Rebuild Notebook

This notebook rebuilds the lap and image manifests from the updated
`lap-image-association.csv`, then searches for a new session-level split.

It does **not** overwrite the existing manifests unless `WRITE_OUTPUTS`
is changed to `True` near the end.

Eligibility rules:

- no incident;
- Sector 1 time below 30 seconds;
- exactly two brake images;
- exactly two throttle images.

Sessions 1–19 are treated as the old sponsor regime and sessions 20+
as the new sponsor regime.

In [2]:
from pathlib import Path
import ast
import re

import numpy as np
import pandas as pd


## 1. Locate the project and updated CSV


In [3]:
possible_project_roots = [
    Path.cwd(),
    Path.cwd().parent,
]

project_root = next(
    (
        candidate.resolve()
        for candidate in possible_project_roots
        if (
            candidate
            / "Data"
            / "collated-data"
            / "lap-image-association.csv"
        ).exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root.\n"
        f"Current working directory: {Path.cwd()}"
    )

data_root = project_root / "Data"

association_csv = (
    data_root
    / "collated-data"
    / "lap-image-association.csv"
)

manifest_dir = data_root / "manifests"
manifest_dir.mkdir(exist_ok=True)

print("Project root:", project_root)
print("Association CSV:", association_csv)
print("Manifest directory:", manifest_dir)


Project root: C:\Users\admin\Desktop\Summer Projects\F1\F1-Optimized-Cornering-Project
Association CSV: C:\Users\admin\Desktop\Summer Projects\F1\F1-Optimized-Cornering-Project\Data\collated-data\lap-image-association.csv
Manifest directory: C:\Users\admin\Desktop\Summer Projects\F1\F1-Optimized-Cornering-Project\Data\manifests


## 2. Load and parse the collated data


In [4]:
def parse_list_cell(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    value = str(value).strip()

    if not value:
        return []

    parsed = ast.literal_eval(value)

    if not isinstance(parsed, list):
        raise ValueError(
            f"Expected a list, received: {parsed!r}"
        )

    return parsed


SESSION_PATTERN = re.compile(
    r"session-(\d+)",
    flags=re.IGNORECASE,
)

LOCAL_LAP_PATTERN = re.compile(
    r"Lap_(\d+)_",
    flags=re.IGNORECASE,
)

HIT_PATTERN = re.compile(
    r"hit_(\d+)",
    flags=re.IGNORECASE,
)


def first_available_path(row):
    if row["related_b_hits"]:
        return row["related_b_hits"][0]

    if row["related_t_hits"]:
        return row["related_t_hits"][0]

    return None


def extract_required_number(
    pattern,
    text,
    description,
):
    if text is None:
        raise ValueError(
            f"Cannot extract {description} from an empty path."
        )

    match = pattern.search(str(text))

    if match is None:
        raise ValueError(
            f"Could not extract {description} from:\n{text}"
        )

    return int(match.group(1))


association_df = pd.read_csv(association_csv)

for column in [
    "related_b_hits",
    "related_t_hits",
    "brake_throttle_deltas",
]:
    association_df[column] = (
        association_df[column]
        .apply(parse_list_cell)
    )

association_df["brake_count"] = (
    association_df["related_b_hits"]
    .apply(len)
)

association_df["throttle_count"] = (
    association_df["related_t_hits"]
    .apply(len)
)

reference_paths = association_df.apply(
    first_available_path,
    axis=1,
)

association_df["session"] = (
    reference_paths.apply(
        lambda path: extract_required_number(
            SESSION_PATTERN,
            path,
            "session number",
        )
    )
)

association_df["session_lap_num"] = (
    reference_paths.apply(
        lambda path: extract_required_number(
            LOCAL_LAP_PATTERN,
            Path(path).name,
            "session-local lap number",
        )
    )
)

association_df["sponsor_regime"] = np.where(
    association_df["session"] <= 19,
    "old_sponsor",
    "new_sponsor",
)

print("Total collated laps:", len(association_df))
print(
    "Sessions:",
    association_df["session"].nunique(),
)


Total collated laps: 255
Sessions: 25


## 3. Apply eligibility rules and calculate classes


In [5]:
eligible_laps = association_df[
    (association_df["incident"] == "none")
    & (association_df["s1_time"] < 30000)
    & (association_df["brake_count"] == 2)
    & (association_df["throttle_count"] == 2)
].copy()

eligible_laps["lap_num"] = (
    eligible_laps["lap_num"].astype(int)
)

q1 = float(
    np.quantile(
        eligible_laps["s1_time"],
        0.25,
    )
)

q3 = float(
    np.quantile(
        eligible_laps["s1_time"],
        0.75,
    )
)


def assign_class(sector_time_ms):
    if sector_time_ms <= q1:
        return "optimal_laps"

    if sector_time_ms < q3:
        return "standard_laps"

    return "sub_standard_laps"


class_to_index = {
    "optimal_laps": 0,
    "standard_laps": 1,
    "sub_standard_laps": 2,
}

eligible_laps["class_name"] = (
    eligible_laps["s1_time"]
    .apply(assign_class)
)

eligible_laps["class_index"] = (
    eligible_laps["class_name"]
    .map(class_to_index)
)

print("Eligible laps:", len(eligible_laps))
print("Eligible images:", len(eligible_laps) * 4)
print("Q1:", q1)
print("Q3:", q3)

display(
    eligible_laps["class_name"]
    .value_counts()
    .reindex(
        list(class_to_index),
        fill_value=0,
    )
    .rename("lap_count")
    .to_frame()
)


Eligible laps: 197
Eligible images: 788
Q1: 28103.0
Q3: 28569.0


,lap_count
class_name,
optimal_laps,50
standard_laps,97
sub_standard_laps,50


## 4. Audit exclusions and new sessions


In [6]:
exclusion_summary = pd.DataFrame({
    "reason": [
        "Incident laps",
        "Sector time >= 30 seconds",
        "Not exactly 2 brake + 2 throttle images",
        "Eligible",
    ],
    "lap_count": [
        int(
            (
                association_df["incident"]
                != "none"
            ).sum()
        ),
        int(
            (
                (association_df["incident"] == "none")
                & (association_df["s1_time"] >= 30000)
            ).sum()
        ),
        int(
            (
                (association_df["incident"] == "none")
                & (association_df["s1_time"] < 30000)
                & (
                    (association_df["brake_count"] != 2)
                    | (
                        association_df[
                            "throttle_count"
                        ] != 2
                    )
                )
            ).sum()
        ),
        len(eligible_laps),
    ],
})

display(exclusion_summary)

new_session_summary = (
    eligible_laps[
        eligible_laps["session"] >= 20
    ]
    .groupby(
        [
            "session",
            "class_name",
        ]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        columns=list(class_to_index),
        fill_value=0,
    )
)

display(new_session_summary)


,reason,lap_count
0,Incident laps,52
1,Sector time >= 30 seconds,1
2,Not exactly 2 brake + 2 throttle images,5
3,Eligible,197


class_name,optimal_laps,standard_laps,sub_standard_laps
session,,,
20,3,4,3
21,1,6,2
22,2,8,2
23,4,8,3
24,8,3,4
25,7,4,3


## 5. Build the 197-row lap manifest


In [7]:
def delta_lookup(delta_entries):
    lookup = {}

    for entry in delta_entries:
        if len(entry) < 3:
            continue

        delta_value = float(entry[0])
        brake_hit = int(entry[1])
        throttle_hit = int(entry[2])

        lookup[
            (brake_hit, throttle_hit)
        ] = delta_value

    return lookup


lap_records = []

for _, row in eligible_laps.iterrows():
    deltas = delta_lookup(
        row["brake_throttle_deltas"]
    )

    delta_1 = deltas.get(
        (1, 1),
        np.nan,
    )

    delta_2 = deltas.get(
        (2, 2),
        np.nan,
    )

    lap_records.append({
        "global_lap_num":
            int(row["lap_num"]),

        "session":
            int(row["session"]),

        "session_lap_num":
            int(row["session_lap_num"]),

        "sponsor_regime":
            row["sponsor_regime"],

        "class_name":
            row["class_name"],

        "class_index":
            int(row["class_index"]),

        "s1_time_ms":
            int(row["s1_time"]),

        "s1_time_seconds":
            float(row["s1_time"]) / 1000,

        "incident":
            row["incident"],

        "delta_1_seconds":
            delta_1,

        "delta_2_seconds":
            delta_2,

        "delta_gap_seconds":
            (
                delta_2 - delta_1
                if (
                    pd.notna(delta_1)
                    and pd.notna(delta_2)
                )
                else np.nan
            ),

        "mean_delta_seconds":
            (
                np.mean(
                    [delta_1, delta_2]
                )
                if (
                    pd.notna(delta_1)
                    and pd.notna(delta_2)
                )
                else np.nan
            ),

        "has_delta_data":
            bool(
                pd.notna(delta_1)
                and pd.notna(delta_2)
            ),
    })

lap_manifest = (
    pd.DataFrame(lap_records)
    .sort_values("global_lap_num")
    .reset_index(drop=True)
)

assert len(lap_manifest) == len(
    eligible_laps
)

assert lap_manifest[
    "global_lap_num"
].is_unique

display(lap_manifest.head())


,global_lap_num,session,session_lap_num,sponsor_regime,class_name,class_index,s1_time_ms,s1_time_seconds,incident,delta_1_seconds,delta_2_seconds,delta_gap_seconds,mean_delta_seconds,has_delta_data
0,3,1,3,old_sponsor,sub_standard_laps,2,28935,28.935,none,NaN,NaN,NaN,NaN,False
1,4,1,4,old_sponsor,sub_standard_laps,2,29188,29.188,none,NaN,NaN,NaN,NaN,False
2,5,1,5,old_sponsor,standard_laps,1,28478,28.478,none,NaN,NaN,NaN,NaN,False
3,6,1,6,old_sponsor,standard_laps,1,28420,28.420,none,NaN,NaN,NaN,NaN,False
4,8,1,8,old_sponsor,sub_standard_laps,2,28812,28.812,none,NaN,NaN,NaN,NaN,False


## 6. Build the image manifest directly from raw images


In [8]:
def normalize_raw_path(raw_path):
    path_text = (
        str(raw_path)
        .replace("\\", "/")
    )

    # Ignore inconsistent ./Data and ../Data prefixes.
    data_marker = "Data/"

    marker_position = path_text.find(
        data_marker
    )

    if marker_position == -1:
        raise ValueError(
            f"Path does not contain Data/:\n{raw_path}"
        )

    relative_from_project = (
        path_text[marker_position:]
    )

    return (
        project_root
        / relative_from_project
    ).resolve()


def hit_number_from_path(raw_path):
    return extract_required_number(
        HIT_PATTERN,
        Path(raw_path).name,
        "hit number",
    )


image_records = []

eligible_by_lap = (
    eligible_laps
    .set_index(
        "lap_num",
        verify_integrity=True,
    )
)

for _, lap_row in lap_manifest.iterrows():
    global_lap_num = int(
        lap_row["global_lap_num"]
    )

    source_row = eligible_by_lap.loc[
        global_lap_num
    ]

    image_groups = [
        (
            "brake",
            source_row["related_b_hits"],
        ),
        (
            "throttle",
            source_row["related_t_hits"],
        ),
    ]

    for signal_type, raw_paths in image_groups:
        for raw_path in raw_paths:
            hit_number = hit_number_from_path(
                raw_path
            )

            resolved_path = normalize_raw_path(
                raw_path
            )

            image_records.append({
                "image_id":
                    (
                        f"lap_{global_lap_num:04d}"
                        f"_{signal_type}"
                        f"_{hit_number:02d}"
                    ),

                # The modelling notebook can use the raw
                # screenshot directly; no class-folder copy
                # is required.
                "cnn_image_path":
                    str(resolved_path),

                "cnn_repo_relative_path":
                    resolved_path.relative_to(
                        project_root
                    ).as_posix(),

                "class_name":
                    lap_row["class_name"],

                "class_index":
                    int(lap_row["class_index"]),

                "global_lap_num":
                    global_lap_num,

                "session":
                    int(lap_row["session"]),

                "session_lap_num":
                    int(
                        lap_row[
                            "session_lap_num"
                        ]
                    ),

                "sponsor_regime":
                    lap_row["sponsor_regime"],

                "signal_type":
                    signal_type,

                "hit_number":
                    hit_number,

                "raw_csv_path":
                    raw_path,

                "raw_image_exists":
                    resolved_path.exists(),

                "s1_time_ms":
                    int(lap_row["s1_time_ms"]),

                "s1_time_seconds":
                    float(
                        lap_row[
                            "s1_time_seconds"
                        ]
                    ),

                "delta_1_seconds":
                    lap_row[
                        "delta_1_seconds"
                    ],

                "delta_2_seconds":
                    lap_row[
                        "delta_2_seconds"
                    ],

                "has_delta_data":
                    bool(
                        lap_row[
                            "has_delta_data"
                        ]
                    ),
            })

image_manifest = (
    pd.DataFrame(image_records)
    .sort_values(
        [
            "global_lap_num",
            "signal_type",
            "hit_number",
        ]
    )
    .reset_index(drop=True)
)

assert len(image_manifest) == (
    4 * len(lap_manifest)
)

assert image_manifest[
    "image_id"
].is_unique

assert (
    image_manifest
    .groupby("global_lap_num")
    .size()
    .eq(4)
    .all()
)

missing_raw_images = image_manifest[
    ~image_manifest[
        "raw_image_exists"
    ]
]

print("Image manifest rows:", len(image_manifest))
print("Missing raw images:", len(missing_raw_images))

if len(missing_raw_images):
    display(
        missing_raw_images[[
            "image_id",
            "raw_csv_path",
            "cnn_image_path",
        ]]
    )


Image manifest rows: 788
Missing raw images: 0


## 7 Search for a session-level split

The search keeps each complete session in one subset and requires:

- all three classes in train, validation and test;
- no session overlap;
- at least one new-sponsor session in each subset.

The score balances:

- target subset sizes;
- class proportions;
- sponsor-regime proportions;
- average Sector 1 time.


In [15]:
train_ratio = 0.70
validation_ratio = 0.15
test_ratio = 0.15

seed = 42
candidate_count = 50000

sessions = np.array(
    sorted(
        lap_manifest["session"].unique()
    )
)

new_sponsor_sessions = set(
    sessions[sessions >= 20]
)

# Determine which complete sessions can safely be used
# for validation and testing.
session_delta_status = (
    lap_manifest
    .groupby("session")["has_delta_data"]
    .all()
)

evaluation_candidate_sessions = np.array(
    sorted(
        session_delta_status[
            session_delta_status
        ].index
    )
)

# Sessions containing any eligible lap without complete
# delta data are forced into training.
forced_train_sessions = set(
    session_delta_status[
        ~session_delta_status
    ].index
)

print(
    "Sessions eligible for validation/test:",
    evaluation_candidate_sessions.tolist(),
)

print(
    "Sessions forced into training:",
    sorted(forced_train_sessions),
)


overall_class_proportions = (
    lap_manifest["class_index"]
    .value_counts(normalize=True)
    .reindex(
        range(3),
        fill_value=0,
    )
)

overall_new_sponsor_proportion = (
    lap_manifest["sponsor_regime"]
    .eq("new_sponsor")
    .mean()
)

overall_mean_time = (
    lap_manifest["s1_time_ms"]
    .mean()
)


def evaluate_assignment(
    train_sessions,
    validation_sessions,
    test_sessions,
):
    split_specs = [
        (
            "train",
            train_sessions,
            train_ratio,
        ),
        (
            "validation",
            validation_sessions,
            validation_ratio,
        ),
        (
            "test",
            test_sessions,
            test_ratio,
        ),
    ]

    score = 0.0
    split_frames = {}

    for (
        split_name,
        split_sessions,
        target_ratio,
    ) in split_specs:

        split_df = lap_manifest[
            lap_manifest["session"]
            .isin(split_sessions)
        ]

        if split_df.empty:
            return None

        # Every split must contain all three classes.
        if set(
            split_df["class_index"].unique()
        ) != {0, 1, 2}:
            return None

        # Every split must contain at least one session
        # recorded under the new sponsor appearance.
        if not (
            set(split_sessions)
            & new_sponsor_sessions
        ):
            return None

        # Validation and test must contain complete delta
        # data for every lap.
        if (
            split_name in {
                "validation",
                "test",
            }
            and not split_df[
                "has_delta_data"
            ].all()
        ):
            return None

        split_frames[
            split_name
        ] = split_df

        actual_ratio = (
            len(split_df)
            / len(lap_manifest)
        )

        size_error = abs(
            actual_ratio
            - target_ratio
        )

        class_proportions = (
            split_df["class_index"]
            .value_counts(normalize=True)
            .reindex(
                range(3),
                fill_value=0,
            )
        )

        class_error = (
            class_proportions
            - overall_class_proportions
        ).abs().sum()

        new_sponsor_proportion = (
            split_df["sponsor_regime"]
            .eq("new_sponsor")
            .mean()
        )

        sponsor_error = abs(
            new_sponsor_proportion
            - overall_new_sponsor_proportion
        )

        time_error = abs(
            split_df["s1_time_ms"].mean()
            - overall_mean_time
        ) / overall_mean_time

        score += (
            4.0 * size_error
            + 2.0 * class_error
            + 1.5 * sponsor_error
            + 2.0 * time_error
        )

    return score, split_frames


rng = np.random.default_rng(seed)

best_result = None
best_assignment = None

# Twenty-five sessions means approximately four sessions
# each for validation and test.
target_validation_sessions = max(
    1,
    round(
        len(sessions)
        * validation_ratio
    ),
)

target_test_sessions = max(
    1,
    round(
        len(sessions)
        * test_ratio
    ),
)

required_evaluation_sessions = (
    target_validation_sessions
    + target_test_sessions
)

if (
    len(evaluation_candidate_sessions)
    < required_evaluation_sessions
):
    raise RuntimeError(
        "There are not enough complete-delta sessions "
        "to create validation and test subsets."
    )


for _ in range(candidate_count):

    shuffled_candidates = (
        rng.permutation(
            evaluation_candidate_sessions
        )
    )

    validation_sessions = set(
        shuffled_candidates[
            :target_validation_sessions
        ]
    )

    test_sessions = set(
        shuffled_candidates[
            target_validation_sessions:
            target_validation_sessions
            + target_test_sessions
        ]
    )

    remaining_complete_sessions = set(
        shuffled_candidates[
            target_validation_sessions
            + target_test_sessions:
        ]
    )

    train_sessions = (
        forced_train_sessions
        | remaining_complete_sessions
    )

    result = evaluate_assignment(
        train_sessions,
        validation_sessions,
        test_sessions,
    )

    if result is None:
        continue

    score, split_frames = result

    if (
        best_result is None
        or score < best_result
    ):
        best_result = score

        best_assignment = (
            train_sessions,
            validation_sessions,
            test_sessions,
            split_frames,
        )


if best_assignment is None:
    raise RuntimeError(
        "No valid session split was found."
    )


(
    train_sessions,
    validation_sessions,
    test_sessions,
    split_frames,
) = best_assignment


print("Best split score:", best_result)

print(
    "Train sessions:",
    sorted(train_sessions),
)

print(
    "Validation sessions:",
    sorted(validation_sessions),
)

print(
    "Test sessions:",
    sorted(test_sessions),
)

Sessions eligible for validation/test: [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]
Sessions forced into training: [1, 2, 3, 4, 5]
Best split score: 0.3444420685961202
Train sessions: [1, 2, 3, 4, 5, np.int64(9), np.int64(10), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(21), np.int64(22), np.int64(23), np.int64(24)]
Validation sessions: [np.int64(11), np.int64(18), np.int64(19), np.int64(20)]
Test sessions: [np.int64(6), np.int64(7), np.int64(8), np.int64(25)]


## 8. Inspect the proposed split


In [16]:
def describe_split(
    split_name,
    split_df,
):
    print("\n" + split_name.upper())
    print("-" * len(split_name))

    print(
        "Sessions:",
        sorted(
            split_df[
                "session"
            ].unique()
        ),
    )

    print("Laps:", len(split_df))
    print("Images:", len(split_df) * 4)

    print(
        "New-sponsor sessions:",
        sorted(
            split_df.loc[
                split_df[
                    "sponsor_regime"
                ] == "new_sponsor",
                "session",
            ].unique()
        ),
    )

    print(
        "Mean Sector 1 time:",
        round(
            split_df[
                "s1_time_ms"
            ].mean(),
            2,
        ),
    )

    print("\nClasses:")

    print(
        split_df["class_name"]
        .value_counts()
        .reindex(
            list(class_to_index),
            fill_value=0,
        )
    )


for split_name in [
    "train",
    "validation",
    "test",
]:
    describe_split(
        split_name,
        split_frames[split_name],
    )



TRAIN
-----
Sessions: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(9), np.int64(10), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(21), np.int64(22), np.int64(23), np.int64(24)]
Laps: 139
Images: 556
New-sponsor sessions: [np.int64(21), np.int64(22), np.int64(23), np.int64(24)]
Mean Sector 1 time: 28371.6

Classes:
class_name
optimal_laps         35
standard_laps        69
sub_standard_laps    35
Name: count, dtype: int64

VALIDATION
----------
Sessions: [np.int64(11), np.int64(18), np.int64(19), np.int64(20)]
Laps: 27
Images: 108
New-sponsor sessions: [np.int64(20)]
Mean Sector 1 time: 28315.85

Classes:
class_name
optimal_laps          7
standard_laps        13
sub_standard_laps     7
Name: count, dtype: int64

TEST
----
Sessions: [np.int64(6), np.int64(7), np.int64(8), np.int64(25)]
Laps: 31
Images: 124
New-sponsor sessions: [np.int64(25)]
Mean Sector 1 time: 28373.35

Classes:
class_name
optimal_laps     

In [17]:
split_comparison = []

for split_name in [
    "train",
    "validation",
    "test",
]:
    split_df = split_frames[
        split_name
    ]

    split_comparison.append({
        "split":
            split_name,

        "sessions":
            split_df[
                "session"
            ].nunique(),

        "laps":
            len(split_df),

        "images":
            len(split_df) * 4,

        "optimal":
            int(
                (
                    split_df[
                        "class_index"
                    ] == 0
                ).sum()
            ),

        "standard":
            int(
                (
                    split_df[
                        "class_index"
                    ] == 1
                ).sum()
            ),

        "sub_standard":
            int(
                (
                    split_df[
                        "class_index"
                    ] == 2
                ).sum()
            ),

        "new_sponsor_laps":
            int(
                split_df[
                    "sponsor_regime"
                ]
                .eq("new_sponsor")
                .sum()
            ),

        "mean_s1_time_ms":
            split_df[
                "s1_time_ms"
            ].mean(),
    })

display(
    pd.DataFrame(
        split_comparison
    )
)


,split,sessions,laps,images,optimal,standard,sub_standard,new_sponsor_laps,mean_s1_time_ms
0,train,17,139,556,35,69,35,51,28371.597122
1,validation,4,27,108,7,13,7,10,28315.851852
2,test,4,31,124,8,15,8,14,28373.354839


## 9. Create split manifests


In [18]:
lap_split_assignments = []

for split_name in [
    "train",
    "validation",
    "test",
]:
    split_df = (
        split_frames[split_name]
        .copy()
    )

    split_df["split"] = split_name

    lap_split_assignments.append(
        split_df
    )

lap_manifest_with_split = (
    pd.concat(
        lap_split_assignments,
        ignore_index=True,
    )
    .sort_values("global_lap_num")
    .reset_index(drop=True)
)

image_manifest_with_split = (
    image_manifest
    .merge(
        lap_manifest_with_split[[
            "global_lap_num",
            "split",
        ]],
        on="global_lap_num",
        how="left",
        validate="many_to_one",
    )
    .sort_values([
        "global_lap_num",
        "signal_type",
        "hit_number",
    ])
    .reset_index(drop=True)
)

assert (
    lap_manifest_with_split[
        "split"
    ].notna().all()
)

assert (
    image_manifest_with_split[
        "split"
    ].notna().all()
)

assert len(
    image_manifest_with_split
) == 4 * len(
    lap_manifest_with_split
)

train_session_set = set(
    lap_manifest_with_split.loc[
        lap_manifest_with_split[
            "split"
        ] == "train",
        "session",
    ]
)

validation_session_set = set(
    lap_manifest_with_split.loc[
        lap_manifest_with_split[
            "split"
        ] == "validation",
        "session",
    ]
)

test_session_set = set(
    lap_manifest_with_split.loc[
        lap_manifest_with_split[
            "split"
        ] == "test",
        "session",
    ]
)

assert train_session_set.isdisjoint(
    validation_session_set
)

assert train_session_set.isdisjoint(
    test_session_set
)

assert validation_session_set.isdisjoint(
    test_session_set
)

print("Split manifest validation passed.")


Split manifest validation passed.


In [19]:
delta_split_summary = (
    lap_manifest_with_split
    .groupby("split")
    .agg(
        total_laps=(
            "global_lap_num",
            "count",
        ),
        laps_with_delta=(
            "has_delta_data",
            "sum",
        ),
    )
)

delta_split_summary[
    "laps_without_delta"
] = (
    delta_split_summary[
        "total_laps"
    ]
    - delta_split_summary[
        "laps_with_delta"
    ]
)

display(delta_split_summary)


evaluation_laps = (
    lap_manifest_with_split[
        lap_manifest_with_split[
            "split"
        ].isin([
            "validation",
            "test",
        ])
    ]
)

assert evaluation_laps[
    "has_delta_data"
].all(), (
    "At least one validation or test lap "
    "does not have complete delta data."
)

assert (
    delta_split_summary.loc[
        "validation",
        "laps_without_delta",
    ] == 0
)

assert (
    delta_split_summary.loc[
        "test",
        "laps_without_delta",
    ] == 0
)

print(
    "Validation and test contain complete "
    "delta data for every lap."
)

,total_laps,laps_with_delta,laps_without_delta
split,,,
test,31,31,0
train,139,111,28
validation,27,27,0


Validation and test contain complete delta data for every lap.


In [21]:
final_split_audit = (
    lap_manifest_with_split
    .groupby("split")
    .agg(
        sessions=("session", "nunique"),
        laps=("global_lap_num", "count"),
        mean_s1_time_ms=("s1_time_ms", "mean"),
        median_s1_time_ms=("s1_time_ms", "median"),
        laps_with_delta=("has_delta_data", "sum"),
        new_sponsor_laps=(
            "sponsor_regime",
            lambda values: (
                values == "new_sponsor"
            ).sum(),
        ),
    )
)

class_counts = (
    lap_manifest_with_split
    .groupby(
        [
            "split",
            "class_name",
        ]
    )
    .size()
    .unstack(fill_value=0)
)

display(final_split_audit)
display(class_counts)

,sessions,laps,mean_s1_time_ms,median_s1_time_ms,laps_with_delta,new_sponsor_laps
split,,,,,,
test,4,31,28373.354839,28321.0,31,14
train,17,139,28371.597122,28297.0,111,51
validation,4,27,28315.851852,28275.0,27,10


class_name,optimal_laps,standard_laps,sub_standard_laps
split,,,
test,8,15,8
train,35,69,35
validation,7,13,7


In [22]:
for split_name in [
    "train",
    "validation",
    "test",
]:
    split_sessions = sorted(
        lap_manifest_with_split.loc[
            lap_manifest_with_split["split"]
            == split_name,
            "session",
        ].unique()
    )

    print(
        f"{split_name}:",
        split_sessions,
    )

train: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(9), np.int64(10), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(21), np.int64(22), np.int64(23), np.int64(24)]
validation: [np.int64(11), np.int64(18), np.int64(19), np.int64(20)]
test: [np.int64(6), np.int64(7), np.int64(8), np.int64(25)]


## 10. Write outputs

Review all previous outputs first.

Set `WRITE_OUTPUTS = True` only when the proposed split looks sensible.
The existing manifest files will be backed up with a `.backup.csv` suffix
before being overwritten.


In [ ]:
WRITE_OUTPUTS = True

output_files = {
    "lap-manifest.csv":
        lap_manifest,

    "image-manifest.csv":
        image_manifest,

    "lap-manifest-with-split.csv":
        lap_manifest_with_split,

    "image-manifest-with-split.csv":
        image_manifest_with_split,
}

if WRITE_OUTPUTS:
    for filename, dataframe in (
        output_files.items()
    ):
        output_path = (
            manifest_dir / filename
        )

        backup_path = (
            manifest_dir
            / filename.replace(
                ".csv",
                ".backup.csv",
            )
        )

        if output_path.exists():
            backup_path.write_bytes(
                output_path.read_bytes()
            )

            print(
                "Backed up:",
                backup_path,
            )

        dataframe.to_csv(
            output_path,
            index=False,
        )

        print("Wrote:", output_path)

else:
    print(
        "Dry run only. No files were overwritten."
    )
